In [0]:
print("Connected to Databricks!")

Connected to Databricks!


In [0]:
print("Spark version:", spark.version)

Spark version: 4.2.0


# Establish the root where our files are stored, start with batch 01 - manual ingestion 

In [0]:
ROOT = ("/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing//landing/"
    "batch_01_arrived_2026-10-05") 

In [0]:
display(dbutils.fs.ls(ROOT))

path,name,size,modificationTime
dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/,master/,0,1792773048000
dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/,online/,0,1792773080000
dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/pos/,pos/,0,1792773070000


In [0]:
for folder in ["master", "online", "pos"]:
    print(f"\n--- {folder.upper()} FILES ---")
    for file in dbutils.fs.ls(f"{ROOT}/{folder}"):
        print(file.name)



--- MASTER FILES ---
product_catalogue_20261005.csv
store_closures.csv
stores.csv

--- ONLINE FILES ---
online_order_lines.csv
online_order_status.csv
online_orders.csv
online_payments.csv

--- POS FILES ---
ST001_T01_20261005.json
ST002_T01_20261005.json


# Ask Spark to read one time in batch mode the data *PRODUCTS*

In [0]:
catalogue_path = (
    f"{ROOT}/master/product_catalogue_20261005.csv"
)
df_catalogue = (
    spark.read
    .option("header", "true") # Use the first row as header
    .option("inferSchema", "true") # Automatically infer data types from the CSV file
    .csv(catalogue_path) # Read the CSV file into a DataFrame
)
display(df_catalogue)


snapshot_date,product_id,product_code,title_id,title,format,product_type,publisher,list_price,status
2026-10-05,P101,9780000001016,T001,The Silent Forest,PAPERBACK,BOOK,North Star Press,40,ACTIVE
2026-10-05,P102,9780000001023,T002,Hidden Truth,PAPERBACK,BOOK,Cedar Books,35,ACTIVE
2026-10-05,P103,9780000001030,T003,The Last Summer,PAPERBACK,BOOK,Cedar Books,30,ACTIVE
2026-10-05,P104,9780000001047,T001,The Silent Forest,HARDCOVER,BOOK,North Star Press,100,ACTIVE
2026-10-05,P105,5940000001056,null,Blue Notebook,null,STATIONERY,null,15,ACTIVE
2026-10-05,P106,5940000001063,null,Chess Board,null,GAME,null,70,ACTIVE
2026-10-05,P107,5940000001070,null,Bookmark Gift Set,null,GIFT,null,20,ACTIVE
2026-10-05,P108,9780000001085,T004,Deep Ocean,HARDCOVER,BOOK,Maple House,90,ACTIVE


In [0]:
df_catalogue.printSchema()
print("Total records:",df_catalogue.count()) # Print the total number of records in the DataFrame


root
 |-- snapshot_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_code: long (nullable = true)
 |-- title_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- format: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- list_price: integer (nullable = true)
 |-- status: string (nullable = true)

Total records: 8


In [0]:
pos_path = f"{ROOT}/pos/ST001_T01_20261005.json" # Path to the POS JSON file

df_pos = (spark.read.option("multiline","true").json(pos_path)) # Read the JSON file into a DataFrame
display(df_pos)
df_pos.printSchema()
print("Total records:",df_pos.count()) # Print the total number of records in the DataFrame

business_date,receipts,store_id,till_id
2026-10-05,"List(List(CASH001, , List(List(0, 9780000001016, , 1, RL-001, 40), List(0, 9780000001023, , 1, RL-002, 35), List(0, 5940000001056, , 1, RL-003, 15)), List(List(90, RON, CARD, SUCCEEDED, PAY-001)), REC001, 2026-10-05T10:00:00, 90, COMPLETED), List(CASH001, , List(List(0, 9780000001016, 3FOR2, 1, RL-004, 40), List(0, 9780000001023, 3FOR2, 1, RL-005, 35), List(30, 9780000001030, 3FOR2, 1, RL-006, 30)), List(List(50, RON, CARD, SUCCEEDED, PAY-002), List(25, RON, CASH, SUCCEEDED, PAY-003)), REC002, 2026-10-05T10:20:00, 75, COMPLETED), List(CASH001, , List(List(0, 9780000001047, , 2, RL-007, 100)), List(), REC003, 2026-10-05T11:00:00, 200, CANCELLED), List(CASH001, REC003, List(List(0, 9780000001047, , 1, RL-008, 100)), List(List(100, RON, CARD, SUCCEEDED, PAY-004)), REC004, 2026-10-05T11:05:00, 100, COMPLETED))",ST001,T01


root
 |-- business_date: string (nullable = true)
 |-- receipts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- cashier_id: string (nullable = true)
 |    |    |-- corrected_from_receipt_id: string (nullable = true)
 |    |    |-- lines: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- discount_amount: long (nullable = true)
 |    |    |    |    |-- product_code: string (nullable = true)
 |    |    |    |    |-- promotion_code: string (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |    |    |-- receipt_line_id: string (nullable = true)
 |    |    |    |    |-- unit_price: long (nullable = true)
 |    |    |-- payments: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- amount: long (nullable = true)
 |    |    |    |    |-- currency: string (nullable = true)
 |    |    |    |    |-- payment_meth

# Changed the inferSchema tag from TRUE to FALSE <Spark identified product_code as long type, changed to string>

In [0]:
df_catalogue = (
    spark.read
    .option("header", True) # Use the first row as header
    .option("inferSchema", False) # Do not automatically infer data types from the CSV file 
    .csv(catalogue_path)
) # Read the CSV file into a DataFrame

df_catalogue.printSchema() # Print the schema of the DataFrame
display(df_catalogue)

root
 |-- snapshot_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- title_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- format: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- list_price: string (nullable = true)
 |-- status: string (nullable = true)



snapshot_date,product_id,product_code,title_id,title,format,product_type,publisher,list_price,status
2026-10-05,P101,9780000001016,T001,The Silent Forest,PAPERBACK,BOOK,North Star Press,40,ACTIVE
2026-10-05,P102,9780000001023,T002,Hidden Truth,PAPERBACK,BOOK,Cedar Books,35,ACTIVE
2026-10-05,P103,9780000001030,T003,The Last Summer,PAPERBACK,BOOK,Cedar Books,30,ACTIVE
2026-10-05,P104,9780000001047,T001,The Silent Forest,HARDCOVER,BOOK,North Star Press,100,ACTIVE
2026-10-05,P105,5940000001056,null,Blue Notebook,null,STATIONERY,null,15,ACTIVE
2026-10-05,P106,5940000001063,null,Chess Board,null,GAME,null,70,ACTIVE
2026-10-05,P107,5940000001070,null,Bookmark Gift Set,null,GIFT,null,20,ACTIVE
2026-10-05,P108,9780000001085,T004,Deep Ocean,HARDCOVER,BOOK,Maple House,90,ACTIVE


# Explode the receipts array and select the nested fields into a new DataFrame

In [0]:
from pyspark.sql.functions import explode

df_receipts_preview = (
    df_pos
    .select(explode("receipts").alias("receipt"))
    .select("receipt.*")
)
display(df_receipts_preview)

cashier_id,corrected_from_receipt_id,lines,payments,receipt_id,receipt_timestamp,receipt_total,status
CASH001,,"List(List(0, 9780000001016, , 1, RL-001, 40), List(0, 9780000001023, , 1, RL-002, 35), List(0, 5940000001056, , 1, RL-003, 15))","List(List(90, RON, CARD, SUCCEEDED, PAY-001))",REC001,2026-10-05T10:00:00,90,COMPLETED
CASH001,,"List(List(0, 9780000001016, 3FOR2, 1, RL-004, 40), List(0, 9780000001023, 3FOR2, 1, RL-005, 35), List(30, 9780000001030, 3FOR2, 1, RL-006, 30))","List(List(50, RON, CARD, SUCCEEDED, PAY-002), List(25, RON, CASH, SUCCEEDED, PAY-003))",REC002,2026-10-05T10:20:00,75,COMPLETED
CASH001,,"List(List(0, 9780000001047, , 2, RL-007, 100))",List(),REC003,2026-10-05T11:00:00,200,CANCELLED
CASH001,REC003,"List(List(0, 9780000001047, , 1, RL-008, 100))","List(List(100, RON, CARD, SUCCEEDED, PAY-004))",REC004,2026-10-05T11:05:00,100,COMPLETED


 # Create the bronze-level Product CATALOGUE DataFrame with source file and ingestion timestamp columns

In [0]:
from pyspark.sql import functions as F

catalogue_path = (
    f"{ROOT}/master/product_catalogue_20261005.csv"
)

df_catalogue_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(catalogue_path)
) # Read the CSV file into a raw DataFrame

df_bronze_catalogue = (
    df_catalogue_raw
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file") # Add a column with the source file path
    )
    .withColumn("_ingested_at", F.current_timestamp()) # Add a column with the current timestamp to indicate when the data was ingested
) 

df_bronze_catalogue.printSchema()
display(df_bronze_catalogue)

root
 |-- snapshot_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- title_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- format: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- list_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)



snapshot_date,product_id,product_code,title_id,title,format,product_type,publisher,list_price,status,_source_file,_ingested_at
2026-10-05,P101,9780000001016,T001,The Silent Forest,PAPERBACK,BOOK,North Star Press,40,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P102,9780000001023,T002,Hidden Truth,PAPERBACK,BOOK,Cedar Books,35,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P103,9780000001030,T003,The Last Summer,PAPERBACK,BOOK,Cedar Books,30,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P104,9780000001047,T001,The Silent Forest,HARDCOVER,BOOK,North Star Press,100,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P105,5940000001056,null,Blue Notebook,null,STATIONERY,null,15,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P106,5940000001063,null,Chess Board,null,GAME,null,70,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P107,5940000001070,null,Bookmark Gift Set,null,GIFT,null,20,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z
2026-10-05,P108,9780000001085,T004,Deep Ocean,HARDCOVER,BOOK,Maple House,90,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:04:29.673Z


 # Write the DataFrame CATALOGUE to the bronze Delta table

In [0]:
bronze_table = (
    "mindit_training.bookstore_ec_bronze."
    "product_catalogue_raw"
) # Define the name of the bronze Delta table

(
    df_bronze_catalogue.write
    .format("delta")
    .mode("error") # Set the write mode to error if the table already exists
    .saveAsTable(bronze_table)
) 

print("Bronze Delta table created successfully!")

Bronze Delta table created successfully!


# Checks on our newly created bronze table Catalogue Products 

In [0]:
df_check = spark.table("mindit_training.bookstore_ec_bronze.product_catalogue_raw") # Load the bronze table into a DataFrame

print("Total records:",df_check.count()) # Print the total number of records in the bronze table
df_check.printSchema() # Print the schema of the bronze table
display(df_check) # Display the contents of the bronze table

Total records: 8
root
 |-- snapshot_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- title_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- format: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- list_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



snapshot_date,product_id,product_code,title_id,title,format,product_type,publisher,list_price,status,_source_file,_ingested_at
2026-10-05,P101,9780000001016,T001,The Silent Forest,PAPERBACK,BOOK,North Star Press,40,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P102,9780000001023,T002,Hidden Truth,PAPERBACK,BOOK,Cedar Books,35,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P103,9780000001030,T003,The Last Summer,PAPERBACK,BOOK,Cedar Books,30,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P104,9780000001047,T001,The Silent Forest,HARDCOVER,BOOK,North Star Press,100,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P105,5940000001056,null,Blue Notebook,null,STATIONERY,null,15,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P106,5940000001063,null,Chess Board,null,GAME,null,70,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P107,5940000001070,null,Bookmark Gift Set,null,GIFT,null,20,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z
2026-10-05,P108,9780000001085,T004,Deep Ocean,HARDCOVER,BOOK,Maple House,90,ACTIVE,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/master/product_catalogue_20261005.csv,2026-10-23T17:06:39.928Z


 # Create the bronze-level POS DataFrame with source file and ingestion timestamp columns

In [0]:
from pyspark.sql import functions as F

pos_path = f"{ROOT}/pos/*.json"

df_pos_raw = (
    spark.read
    .option("multiLine", "true")
    .json(pos_path) # Read the JSON files into a raw DataFrame
)

df_bronze_pos = (
    df_pos_raw
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file") # Add a column with the source file path
    )
    .withColumn("_ingested_at", F.current_timestamp()) # Add a column with the current timestamp to indicate when the data was ingested
)

df_bronze_pos.printSchema()

root
 |-- business_date: string (nullable = true)
 |-- receipts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- cashier_id: string (nullable = true)
 |    |    |-- corrected_from_receipt_id: string (nullable = true)
 |    |    |-- lines: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- discount_amount: long (nullable = true)
 |    |    |    |    |-- product_code: string (nullable = true)
 |    |    |    |    |-- promotion_code: string (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |    |    |-- receipt_line_id: string (nullable = true)
 |    |    |    |    |-- unit_price: long (nullable = true)
 |    |    |-- payments: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- amount: long (nullable = true)
 |    |    |    |    |-- currency: string (nullable = true)
 |    |    |    |    |-- payment_meth

In [0]:
display(
    df_bronze_pos.select(
        "store_id",
        "business_date",
        F.size("receipts").alias("receipt_count"),
        "_source_file"
    ) # Display a preview of the POS data with receipt counts and source file information
)

print("POS files:", df_bronze_pos.count()) # Print the total number of POS records

store_id,business_date,receipt_count,_source_file
ST001,2026-10-05,4,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/pos/ST001_T01_20261005.json
ST002,2026-10-05,1,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/pos/ST002_T01_20261005.json


POS files: 2


# Write the POS DataFrame to the Bronze Delta table

In [0]:
pos_bronze_table = (
    "mindit_training.bookstore_ec_bronze.pos_raw"
)

(
    df_bronze_pos.write
    .format("delta")
    .mode("error") # Set the write mode to error if the table already exists
    .saveAsTable(pos_bronze_table)
) 

print("POS Bronze table created successfully!")

POS Bronze table created successfully!


# Display a preview of the POS bronze table with receipt counts, source file, and ingestion timestamp

In [0]:
from pyspark.sql import functions as F

pos_bronze_table = (
    "mindit_training.bookstore_ec_bronze.pos_raw"
)

df_pos_check = spark.table(pos_bronze_table) # Load the POS bronze table into a DataFrame

display(
    df_pos_check.select(
        "store_id",
        "business_date",
        F.size("receipts").alias("receipt_count"),
        "_source_file",
        "_ingested_at"
    ) # Display a preview of the POS bronze table with receipt counts, source file, and ingestion timestamp
)

print("Total POS files:", df_pos_check.count())

print(
    "Total receipts:",
    df_pos_check
    .select(F.explode("receipts").alias("receipt"))
    .count()
)

store_id,business_date,receipt_count,_source_file,_ingested_at
ST001,2026-10-05,4,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/pos/ST001_T01_20261005.json,2026-10-23T17:16:14.465Z
ST002,2026-10-05,1,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/pos/ST002_T01_20261005.json,2026-10-23T17:16:14.465Z


Total POS files: 2
Total receipts: 5


 # Create the bronze-level Ornline Orders DataFrame with source file and ingestion timestamp columns

In [0]:
from pyspark.sql import functions as F

online_orders_path = f"{ROOT}/online/online_orders.csv"

df_online_orders_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(online_orders_path)
) # Read the online orders CSV file into a raw DataFrame

df_bronze_online_orders = (
    df_online_orders_raw
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file") # Add a column with the source file path
    )
    .withColumn("_ingested_at", F.current_timestamp()) # Add a column with the current timestamp to indicate when the data was ingested
)

df_bronze_online_orders.printSchema()
display(df_bronze_online_orders) # Display a preview of the online orders bronze DataFrame

print("Total online orders:", df_bronze_online_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- placed_at: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- shipping_fee: string (nullable = true)
 |-- order_total: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)



order_id,customer_id,placed_at,delivery_city,shipping_fee,order_total,_source_file,_ingested_at
O5001,null,2026-10-05T11:00:00,Cluj-Napoca,10,85,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:21:29.925Z
O5002,null,2026-10-05T12:00:00,Bistrita,10,55,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:21:29.925Z
O5003,null,2026-10-05T13:00:00,Suceava,10,110,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:21:29.925Z


Total online orders: 3


# Write the ONLINE ORDERS bronze DataFrame to the Delta table

In [0]:
online_orders_bronze_table = (
    "mindit_training.bookstore_ec_bronze."
    "online_orders_raw"
) # Define the name of the online orders bronze table

(
    df_bronze_online_orders.write
    .format("delta")
    .mode("error") # Set the write mode to error if the table already exists
    .saveAsTable(online_orders_bronze_table)
) # Write the online orders bronze DataFrame to the Delta table

print("Online Orders Bronze table created successfully!")

Online Orders Bronze table created successfully!


In [0]:
df_orders_check = spark.table(online_orders_bronze_table) # Load the online orders bronze table into a DataFrame

print("Online order records:", df_orders_check.count()) # Print the total number of online order records
display(df_orders_check) # Display a preview of the online orders bronze table

Online order records: 3


order_id,customer_id,placed_at,delivery_city,shipping_fee,order_total,_source_file,_ingested_at
O5001,null,2026-10-05T11:00:00,Cluj-Napoca,10,85,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:22:44.801Z
O5002,null,2026-10-05T12:00:00,Bistrita,10,55,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:22:44.801Z
O5003,null,2026-10-05T13:00:00,Suceava,10,110,dbfs:/Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing/batch_01_arrived_2026-10-05/online/online_orders.csv,2026-10-23T17:22:44.801Z


# Define an universal function to Ingest a CSV file into a bronze Delta table.

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "mindit_training.bookstore_ec_bronze"

def ingest_csv_to_bronze(file_name, table_name): 
    """Ingest a CSV file into a bronze Delta table.

    Args:
        file_name (str): The name of the CSV file to ingest.
        table_name (str): The name of the target bronze Delta table.
    """

    source_path = f"{ROOT}/online/{file_name}"
    target_table = f"{BRONZE_SCHEMA}.{table_name}"

    # Prevent accidental duplicate initial loads
    if spark.catalog.tableExists(target_table): 
        print(f"Table already exists: {target_table}")
        return

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false") # Do not infer schema automatically, treat all columns as STRINGS
        .csv(source_path)
    ) # Read the CSV file into a DataFrame

    df_bronze = ( 
        df.select(
            "*",
            F.col("_metadata.file_path").alias("_source_file") 
        )
        .withColumn("_ingested_at", F.current_timestamp())
    ) # Add source file and ingestion timestamp columns to the DataFrame

    (
        df_bronze.write 
        .format("delta")
        .mode("error") # Set the write mode to error if the table already exists
        .saveAsTable(target_table) # saved in Unity Catalog
    ) # Write the bronze DataFrame to the Delta table

    print(f"Created: {target_table}") # Print the name of the created table
    print(f"Records: {spark.table(target_table).count()}") # Print the number of records in the created table

# Ingest each CSV file into its corresponding bronze Delta table

In [0]:
online_sources = {
    "online_order_lines.csv": "online_order_lines_raw",
    "online_order_status.csv": "online_order_status_raw",
    "online_payments.csv": "online_payments_raw"
} # Dictionary mapping CSV file names to their corresponding bronze Delta table names

for file_name, table_name in online_sources.items():
    ingest_csv_to_bronze(file_name, table_name) 

Created: mindit_training.bookstore_ec_bronze.online_order_lines_raw
Records: 5
Created: mindit_training.bookstore_ec_bronze.online_order_status_raw
Records: 7
Created: mindit_training.bookstore_ec_bronze.online_payments_raw
Records: 5


In [0]:
online_tables = [
    "online_orders_raw",
    "online_order_lines_raw",
    "online_order_status_raw",
    "online_payments_raw"
]

for table_name in online_tables:
    df = spark.table(f"{BRONZE_SCHEMA}.{table_name}")
    print(f"{table_name}: {df.count()} records")
    df.printSchema() 

online_orders_raw: 3 records
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- placed_at: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- shipping_fee: string (nullable = true)
 |-- order_total: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

online_order_lines_raw: 5 records
root
 |-- order_line_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

online_order_status_raw: 7 records
root
 |-- order_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- status_event_timestamp: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingeste

# Ingest a MASTER CSV file into a bronze Delta table

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "mindit_training.bookstore_ec_bronze"

def ingest_master_to_bronze(file_name, table_name): 
    """Ingest a master CSV file into a bronze Delta table.

    Args:
        file_name (str): The name of the CSV file to ingest.
        table_name (str): The name of the target bronze Delta table.
    """

    source_path = f"{ROOT}/master/{file_name}"
    target_table = f"{BRONZE_SCHEMA}.{table_name}"

    if spark.catalog.tableExists(target_table):
        print(f"Already exists: {target_table}")
        return

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false") # Do not infer schema automatically, treat all columns as STRINGS
        .csv(source_path)
    )

    df_bronze = (
        df.select(
            "*",
            F.col("_metadata.file_path").alias("_source_file")
        )
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        df_bronze.write
        .format("delta")
        .mode("error")
        .saveAsTable(target_table)
    )

    print(f"Created: {target_table}")
    print(f"Records: {spark.table(target_table).count()}")

In [0]:
master_sources = {
    "stores.csv": "stores_raw",
    "store_closures.csv": "store_closures_raw"
}

for file_name, table_name in master_sources.items():
    ingest_master_to_bronze(file_name, table_name)

Created: mindit_training.bookstore_ec_bronze.stores_raw
Records: 3
Created: mindit_training.bookstore_ec_bronze.store_closures_raw
Records: 1


# Count our records for the created Delta Tables

In [0]:
bronze_tables = [
    "product_catalogue_raw",
    "stores_raw",
    "store_closures_raw",
    "pos_raw",
    "online_orders_raw",
    "online_order_lines_raw",
    "online_order_status_raw",
    "online_payments_raw"
] # List of all bronze Delta tables to check record counts for

for table_name in bronze_tables:

    full_name = f"{BRONZE_SCHEMA}.{table_name}"
    count = spark.table(full_name).count()

    print(f"{table_name}: {count} records")

product_catalogue_raw: 8 records
stores_raw: 3 records
store_closures_raw: 1 records
pos_raw: 2 records
online_orders_raw: 3 records
online_order_lines_raw: 5 records
online_order_status_raw: 7 records
online_payments_raw: 5 records


In [ ]:
%sql
SELECT current_catalog() AS catalog_name; 

SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema LIKE 'bookstore_ec%'
ORDER BY table_schema, table_name;

SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema LIKE 'bookstore_ec%'
ORDER BY table_schema, table_name, ordinal_position; 

: 